# Conversation Memory in LangChain

This notebook shows why a basic LLM call is stateless, then adds per-session chat history with `RunnableWithMessageHistory`.

In [76]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

In [77]:
# Load OPENAI_API_KEY and any other values from the local .env file.
load_dotenv()

True

In [78]:
# A normal ChatOpenAI instance does not remember previous calls by itself.
model = ChatOpenAI(model='gpt-4o', temperature=0)

In [79]:
parser = StrOutputParser()

In [80]:
model.invoke('Hello, I am Arun')

AIMessage(content='Hello, Arun! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 12, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_e160a45017', 'id': 'chatcmpl-EEAqaXklcbOAWATx8zsb6m005UGcj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0144d-276f-7591-a4e3-d2d3125ee539-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 11, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Why Memory Is Needed

- A normal LLM API call is stateless.
- If you ask the model for your name in a later call, it will not know unless the earlier messages are sent again.
- LangChain memory helpers manage those previous messages for each session.

## `RunnableWithMessageHistory`

In [86]:
# from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [87]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [88]:
# Create one in-memory history object and add messages manually.
# message_hist = InMemoryChatMessageHistory()
message_hist = ChatMessageHistory()

In [89]:
message_hist.add_user_message("hello")
message_hist.add_ai_message("Hi there!")

In [90]:
print(message_hist)

Human: hello
AI: Hi there!


In [91]:
# Store one chat history per session id.
store = {}

In [95]:
def get_history(session_id: str):
    """Return the chat history for a session, creating it on first use."""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [96]:
# Use the same model settings for every turn in the remembered conversation.
llm = ChatOpenAI(model='gpt-4o', temperature=0.2)

In [97]:
parser = StrOutputParser()

In [98]:
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful AI assistant.'),
    MessagesPlaceholder(variable_name="history"),
    ('human', '{input}'),
])

In [99]:
# The prompt receives history, the LLM generates a response, and the parser returns plain text.
chain = prompt | llm | parser

In [100]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key='input',
    history_messages_key='history',
)

In [101]:
response1 = chain_with_history.invoke(
    {'input': 'Hello, I am Arun. How are you?'},
    config={'configurable': {'session_id': 'arun_001'}},
)
response1

"Hello, Arun! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with whatever you need. How can I assist you today?"

In [102]:
response2 = chain_with_history.invoke(
    {'input': 'What did I just say?'},
    config={'configurable': {'session_id': 'arun_001'}},
)
response2

"You introduced yourself as Arun and asked how I am. Is there anything specific you'd like to know or discuss?"